## Mapping
`codegen_mapping_strategy` in the machine file picks a GeMM's tile: `heuristic`, `autotune`,
`external-then-heuristic` or `external-then-autotune`.

The example is a batched GeMM (2 x 1024 x 1024 x 1024).

In [ ]:
!grep -H -E '^(num_cores|pytorchsim_|codegen_)' togsim_configs/togsim_config_heuristic.yml togsim_configs/togsim_config_external_mapping.yml togsim_configs/togsim_config_autotune.yml

In [ ]:
import os
os.environ["TORCHSIM_TARGET"] = os.path.join(os.getcwd(), "togsim_configs", "togsim_config_heuristic.yml")
os.environ["TORCHSIM_LOG_PATH"] = os.path.join(os.getcwd(), "togsim_results")
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "outputs")
import glob
import torch

def latest_log():
    """The newest TOGSim log under TORCHSIM_LOG_PATH."""
    return max(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")), key=os.path.getmtime)

_seen = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")))

def new_logs():
    """The TOGSim logs written since the last call, oldest first: one per simulation, two for a
    resident mm (its operand's copy-in, then its body), whose cycles add up."""
    logs = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log"))) - _seen
    _seen.update(logs)
    return sorted(logs, key=os.path.getmtime)

### Default mapping is based on heuristic.

In [ ]:
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "heuristic")
device = torch.device("npu:0")

input = torch.randn(2, 1024, 1024, dtype=torch.bfloat16).to(device=device)
weight = torch.randn(2, 1024, 1024, dtype=torch.bfloat16).to(device=device)

opt_fn = torch.compile(dynamic=False)(torch.bmm)
npu_outut = opt_fn(input, weight)

In [ ]:
for log in new_logs():
    !grep "Total execution cycles" {log}

### External Mapping
User can set tile size manually from external file: `codegen_external_mapping_file` (relative to the
machine file) maps `"M_N_K"` (a bmm's per batch) to `TILE_M`, `TILE_N`, `TILE_K`.

In [ ]:
!cat tutorial_external_mapping.json

In [ ]:
torch._dynamo.reset()
os.environ["TORCHSIM_TARGET"] = os.path.join(os.getcwd(), "togsim_configs", "togsim_config_external_mapping.yml")
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "external")

input = torch.randn(2, 1024, 1024, dtype=torch.bfloat16).to(device=device)
weight = torch.randn(2, 1024, 1024, dtype=torch.bfloat16).to(device=device)

opt_fn = torch.compile(dynamic=False)(torch.bmm)
npu_outut = opt_fn(input, weight)

In [ ]:
for log in new_logs():
    !grep "Total execution cycles" {log}

### Autotune
In timing mode the top candidates (4 by default, `codegen_autotune_template_topk`) are each timed on
TOGSim and the fastest is kept.

In [ ]:
torch._dynamo.reset()
os.environ["TORCHSIM_TARGET"] = os.path.join(os.getcwd(), "togsim_configs", "togsim_config_autotune.yml")
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "autotune")

input = torch.randn(2, 1024, 1024, dtype=torch.bfloat16).to(device=device)
weight = torch.randn(2, 1024, 1024, dtype=torch.bfloat16).to(device=device)

opt_fn = torch.compile(dynamic=False)(torch.bmm)
npu_outut = opt_fn(input, weight)

In [ ]:
for log in new_logs():
    !grep "Total execution cycles" {log}